# Агрегаты из bureau и previous_application
Нужны файлы в data/raw/: application_train.csv, bureau.csv, previous_application.csv
Нужен lightgbm: pip install lightgbm
Ноутбук лежит в notebooks/, код берём из src/

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

sys.path.insert(0, "..")  # корень проекта на уровень выше notebooks/
from src.features import ID_COL, TARGET_COL, build_dataset
from src.models import make_logreg, cv_auc, to_lgbm, make_lgbm

DATA = Path("../data/raw")

In [ ]:
app = pd.read_csv(DATA / "application_train.csv")
bureau = pd.read_csv(DATA / "bureau.csv")
prev = pd.read_csv(DATA / "previous_application.csv")
print("app:", app.shape, "| bureau:", bureau.shape, "| prev:", prev.shape)

## 1. Покрытие: у какой доли клиентов вообще есть история?

In [ ]:
print(f"Есть в bureau:   {app[ID_COL].isin(bureau[ID_COL]).mean():.1%}")
print(f"Есть в previous: {app[ID_COL].isin(prev[ID_COL]).mean():.1%}")

print("\nТипичное число кредитов на клиента:")
print(bureau.groupby(ID_COL).size().describe().round(1))

In [ ]:
# Аномалии в bureau / previous
print(bureau["CREDIT_ACTIVE"].value_counts())
print("\nСтатусы прошлых заявок:")
print(prev["NAME_CONTRACT_STATUS"].value_counts(normalize=True).round(3))
print("\nЗаглушка 365243 в previous:")
for c in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
    print(f"  {c}: {(prev[c] == 365243).mean():.1%}")

## 2. Собираем датасет

In [ ]:
full = build_dataset(app, bureau, prev)
base = build_dataset(app)  # только основная таблица, для честного сравнения
print("base:", base.shape, "| full:", full.shape)
print("Строк не потеряно:", len(full) == len(app), "| ID уникальны:", full[ID_COL].is_unique)

In [ ]:
# Дефолт у клиентов с историей и без
for flag in ["HAS_BUREAU", "HAS_PREV"]:
    print(full.groupby(flag)[TARGET_COL].agg(["mean", "count"]).round(4), "\n")

In [ ]:
# Топ-15 новых признаков по |корреляции| с таргетом
new_cols = [c for c in full.columns if c not in base.columns]
corr = full[new_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(15).round(3))

## 3. Лесенка улучшений: логрег и LightGBM на одних и тех же фолдах

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
y = full[TARGET_COL]


def prepare(df):
    X = df.drop(columns=[TARGET_COL, ID_COL])
    return X


X_base, X_full = prepare(base), prepare(full)


# def cv_auc(model, X):
#     return cross_val_score(model, X, y, cv=skf, scoring="roc_auc")

In [ ]:
# Логрег (пайплайн из src/models.py)
s_lr_base = cv_auc(make_logreg(X_base), X_base)
s_lr_full = cv_auc(make_logreg(X_full), X_full)
print(f"LogReg base: {s_lr_base.mean():.4f} | full: {s_lr_full.mean():.4f}")

In [ ]:
# LightGBM: категории -> dtype 'category', пропуски обрабатывает сам
from lightgbm import LGBMClassifier


# def to_lgbm(X):
#     X = X.copy()
#     for c in X.select_dtypes("object").columns:
#         X[c] = X[c].astype("category")
#     return X


lgbm = LGBMClassifier(
    n_estimators=400, learning_rate=0.06, num_leaves=31,
    colsample_bytree=0.5, subsample=0.8, subsample_freq=1,
    random_state=42, verbose=-1, n_jobs=-1,
)
# ~ несколько минут на каждый вариант
s_gb_base = cv_auc(lgbm, to_lgbm(X_base))
s_gb_full = cv_auc(lgbm, to_lgbm(X_full))

In [ ]:
scores = {
    "LogReg: app only": s_lr_base,
    "LogReg: + bureau + prev": s_lr_full,
    "LightGBM: app only": s_gb_base,
    "LightGBM: + bureau + prev": s_gb_full,
}
summary = pd.DataFrame({
    "mean AUC": {k: v.mean() for k, v in scores.items()},
    "std": {k: v.std() for k, v in scores.items()},
}).round(4)
summary["Gini"] = (2 * summary["mean AUC"] - 1).round(4)
summary

Ожидание: логрег прибавит немного, а LightGBM заметно сильнее (порядка +0.02 AUC).
Если это так, значит основной прирост даёт именно сочетание богатых признаков и модели,
которая умеет с ними работать. Это и есть главный вывод для README.